In [0]:
%python
# Check店铺范围
dbutils.widgets.text('dt','2026-03-09')
dbutils.widgets.text('start_mth','2025-01')
dbutils.widgets.text('end_mth','2025-12')
dbutils.widgets.text(
  "store", 
  """
	'JD - UL','JD - UL CBE','JD - AHC','JD - AHC CBE'    
	,'TMALL FGS - BPC','TMALL FGS - VSL','TMALL FGS - AHC'
	,'TMALL FGS - CLEAR','TMALL FGS - NEXXUS'
	,'TMALL FGS - PONDS','TMALL FGS - AHC CBE','TMALL FGS - BPC CBE' 
	
	,'TMALL SUPER - UL','TMALL SUPER'
	,'PDD - UL','PDD - AHC','PDD - AHC DTC','PDD - VSL'    
	,'VIP - UL','VIP - AHC','VIP - AHC CBE'    
	,'DEWU - AHC'
	,'PDD 间供' 
	,'TMALL FGS - ZHONGHUA'    
  ,'TMALL FGS - REGENERATE'    
	,'TMALL FGS - HC'
  ,'TMALL FGS - LUX'   
  """
)

start_mth = dbutils.widgets.get('start_mth')
end_mth = dbutils.widgets.get('end_mth')


In [0]:
%sql
select distinct dt from ${spark.var.catalog.ads}.fin_cpt.ads_second_total_sales_mf
order by dt desc

dt
2026-11-10
2026-11-09
2026-10-13
2026-10-12
2026-09-08
2026-08-08
2026-07-10
2026-06-06
2026-06-05
2026-05-12


In [0]:
select * from ${spark.var.catalog.ads}.fin_cpt.ads_second_total_sales_mf
where dt = '2026-11-10' 
    and bu_name = 'BEAUTY&WELLBEING'
    and store_name = 'TMALL FGS - BPC'
-- order by dt desc

month,bu_code,bu_name,customer_type_code,customer_type,site_code,site_name,sales_type,store_code,store_name,category_code,category_name,brand_code,brand_name,product_line_group_code,product_line_group,secondary_after_gmv_amount,secondary_gmv_amount,create_by,dw_create_time,dw_last_update_time,dw_batch_num,dt
202602,,BEAUTY&WELLBEING,,EC,,TMALL FGS,2C,,TMALL FGS - BPC,,HAIR,,DOVE,,DREAM,228524.270000,250402.430000,Automatic,2026-11-10T17:08:34Z,2026-11-10T17:08:34Z,20261110170834,2026-11-10
202608,,BEAUTY&WELLBEING,,EC,,TMALL FGS,2C,,TMALL FGS - BPC,,HAIR,,DOVE,,NOVA,1114142.360000,1493373.380000,Automatic,2026-11-10T17:08:34Z,2026-11-10T17:08:34Z,20261110170834,2026-11-10
202607,,BEAUTY&WELLBEING,,EC,,TMALL FGS,2C,,TMALL FGS - BPC,,HAIR,,CLEAR,,CATALYST,51263.900000,59675.210000,Automatic,2026-11-10T17:08:34Z,2026-11-10T17:08:34Z,20261110170834,2026-11-10
202608,,BEAUTY&WELLBEING,,EC,,TMALL FGS,2C,,TMALL FGS - BPC,,HAIR,,CLEAR,,CALENDULA,87333.220000,105988.100000,Automatic,2026-11-10T17:08:34Z,2026-11-10T17:08:34Z,20261110170834,2026-11-10
202607,,BEAUTY&WELLBEING,,EC,,TMALL FGS,2C,,TMALL FGS - BPC,,HAIR,,CLEAR,,CALIBER,160411.010000,210664.050000,Automatic,2026-11-10T17:08:34Z,2026-11-10T17:08:34Z,20261110170834,2026-11-10
202601,,BEAUTY&WELLBEING,,EC,,TMALL FGS,2C,,TMALL FGS - BPC,,HAIR,,DOVE,,DEBEERS,7079.360000,8427.060000,Automatic,2026-11-10T17:08:34Z,2026-11-10T17:08:34Z,20261110170834,2026-11-10
202604,,BEAUTY&WELLBEING,,EC,,TMALL FGS,2C,,TMALL FGS - BPC,,HAIR,,CLEAR,,CALIBER,192212.380000,211381.690000,Automatic,2026-11-10T17:08:34Z,2026-11-10T17:08:34Z,20261110170834,2026-11-10
202601,,BEAUTY&WELLBEING,,EC,,TMALL FGS,2C,,TMALL FGS - BPC,,HAIR,,DOVE,,DREAM,35774.250000,38544.330000,Automatic,2026-11-10T17:08:34Z,2026-11-10T17:08:34Z,20261110170834,2026-11-10
202606,,BEAUTY&WELLBEING,,EC,,TMALL FGS,2C,,TMALL FGS - BPC,,HAIR,,DOVE,,DOGMA,571627.920000,669009.900000,Automatic,2026-11-10T17:08:34Z,2026-11-10T17:08:34Z,20261110170834,2026-11-10
202606,,BEAUTY&WELLBEING,,EC,,TMALL FGS,2C,,TMALL FGS - BPC,,HAIR,,LUX,,LUSTER,63161.230000,81454.460000,Automatic,2026-11-10T17:08:34Z,2026-11-10T17:08:34Z,20261110170834,2026-11-10


In [0]:
-- 二级gmv raw_data, By month+store+brand+product_line_group
create or replace table ${spark.var.catalog.tmp}.fin.tmp_secondary_gmv_check
select
    coalesce(t1.month,t2.month) as month
    ,coalesce(t1.store_name,t2.store_name) as store_name
    -- ,'BEAUTY&WELLBEING' as bu
    -- ,t1.bu_name as bu
    ,coalesce(t1.bu_name,t2.bu) as bu
    ,coalesce(t1.category_name,t2.category_name) as category_name
    ,coalesce(t1.brand_name,t2.brand_name) as brand_name
    ,coalesce(t1.product_line_group,t2.product_line_group) as product_line_group
    ,coalesce(round(t1.gmv_after_refund,2),0)  as item_gmv_after_refund
    ,coalesce(round(t1.gmv             ,2),0)  as item_gmv
    ,coalesce(round(t2.gmv             ,2),0)  as sku_gmv
    ,coalesce(round(t2.both_gmv        ,2),0)  as both_gmv
    ,coalesce(round(t2.jz_gmv          ,2),0)  as jz_gmv
    ,coalesce(round(t2.sku_gmv         ,2),0)  as ul_sku_gmv
    ,case when t1.store_name like 'TMALL FGS%' then round(nvl(t1.gmv,0) -nvl(t2.gmv,0),0)
          else round(nvl(t1.gmv_after_refund,0) -nvl(t2.gmv,0),0) end as item_sku_gap 
from (
  select
    month
    ,store_name
    ,category_name
    ,brand_name
    ,product_line_group
    ,secondary_after_gmv_amount as gmv_after_refund
    ,secondary_gmv_amount       as gmv
    ,bu_name
  from ${spark.var.catalog.ads}.fin_cpt.ads_second_total_sales_mf
  where dt = '$dt' and abs(secondary_gmv_amount) > 0
  and sales_type='2C'

) t1
full join (


  select
    replace(left(period_key,7),'-','') as month
    ,customer as store_name
    ,bu
    ,category as category_name
    ,brand as brand_name
    ,product_line_group
    ,sum(case when both_flag = 'N' then 0 else gmv end)           as both_gmv
    ,sum(case when jz_flag   = 'N' then 0 else gmv end)           as jz_gmv
    ,sum(case when sku_flag  = 'N' then 0 else gmv end)           as sku_gmv
    ,sum(gmv)                                                     as gmv
    ,sum(case when both_flag = 'N' then 0 else gmv end)/sum(gmv)  as both_coverage
    ,sum(case when jz_flag   = 'N' then 0 else gmv end)/sum(gmv)  as jz_coverage
    ,sum(case when sku_flag  = 'N' then 0 else gmv end)/sum(gmv)  as sku_coverage
  from (
    select 
      period_key
      -- ,customer
      ,case  when customer='JD - REGENERATE' then 'JD - UL'  else customer  end as customer --20260907
      ,bu
      ,category
      ,brand
      ,product_line_group
      ,case when flag = 1 then 'Y' else 'N' end as jz_flag  --    `是否有机制打标`
      ,case when sku_code = '' or sku_code is null then 'N' else 'Y' end as sku_flag -- `是否有SKU Mapping`
      ,case when flag = 1 and sku_code <> '' and sku_code is not null then 'Y' else 'N' end as both_flag
      ,gmv
    from ${spark.var.catalog.tmp}.fin.tmp_trd_total_item_sales_all
    where abs(gmv) > 0
  )
  group by left(period_key,7)
          ,customer
          ,bu
          ,category
          ,brand
          ,product_line_group



          
) t2
on t1.month = t2.month and t1.store_name = t2.store_name and t1.category_name = t2.category_name and t1.brand_name = t2.brand_name
and t1.product_line_group = t2.product_line_group and t1.bu_name = t2.bu
where coalesce(t1.store_name,t2.store_name) in ($store)
-- and coalesce(t1.category_name,t2.category_name) in ('SKIN','HAIR')
and coalesce(t1.bu_name,t2.bu) in ('BEAUTY&WELLBEING','PERSONAL CARE','HOME CARE') --检查3个bu的数据
;

num_affected_rows,num_inserted_rows


In [0]:
-- 二级gmv gap, By month+store+bu
select
    store_name,bu
    ,round(sum(if(month = '202601',both_gmv, null)) /sum(if(month = '202601',sku_gmv, null)),8) as mth01_both_gmv_coverage
    ,round(sum(if(month = '202602',both_gmv, null)) /sum(if(month = '202602',sku_gmv, null)),8) as mth02_both_gmv_coverage
    ,round(sum(if(month = '202603',both_gmv, null)) /sum(if(month = '202603',sku_gmv, null)),8) as mth03_both_gmv_coverage
    ,round(sum(if(month = '202604',both_gmv, null)) /sum(if(month = '202604',sku_gmv, null)),8) as mth04_both_gmv_coverage
    ,round(sum(if(month = '202605',both_gmv, null)) /sum(if(month = '202605',sku_gmv, null)),8) as mth05_both_gmv_coverage
    ,round(sum(if(month = '202606',both_gmv, null)) /sum(if(month = '202606',sku_gmv, null)),8) as mth06_both_gmv_coverage
    ,round(sum(if(month = '202607',both_gmv, null)) /sum(if(month = '202607',sku_gmv, null)),8) as mth07_both_gmv_coverage
    ,round(sum(if(month = '202608',both_gmv, null)) /sum(if(month = '202608',sku_gmv, null)),8) as mth08_both_gmv_coverage
    ,round(sum(if(month = '202609',both_gmv, null)) /sum(if(month = '202609',sku_gmv, null)),8) as mth09_both_gmv_coverage
    ,round(sum(if(month = '202610',both_gmv, null)) /sum(if(month = '202610',sku_gmv, null)),8) as mth10_both_gmv_coverage
    ,round(sum(if(month = '202611',both_gmv, null)) /sum(if(month = '202611',sku_gmv, null)),8) as mth11_both_gmv_coverage
    ,round(sum(if(month = '202612',both_gmv, null)) /sum(if(month = '202612',sku_gmv, null)),8) as mth12_both_gmv_coverage
    ,round(sum(both_gmv)/sum(sku_gmv),8)                                                        as ytd_both_gmv_coverage
    ,''                                                                                         as interval_1
    ,round(sum(if(month = '202601',jz_gmv, null)) /sum(if(month = '202601',sku_gmv, null)),8)   as mth01_jz_gmv_coverage
    ,round(sum(if(month = '202602',jz_gmv, null)) /sum(if(month = '202602',sku_gmv, null)),8)   as mth02_jz_gmv_coverage
    ,round(sum(if(month = '202603',jz_gmv, null)) /sum(if(month = '202603',sku_gmv, null)),8)   as mth03_jz_gmv_coverage
    ,round(sum(if(month = '202604',jz_gmv, null)) /sum(if(month = '202604',sku_gmv, null)),8)   as mth04_jz_gmv_coverage
    ,round(sum(if(month = '202605',jz_gmv, null)) /sum(if(month = '202605',sku_gmv, null)),8)   as mth05_jz_gmv_coverage
    ,round(sum(if(month = '202606',jz_gmv, null)) /sum(if(month = '202606',sku_gmv, null)),8)   as mth06_jz_gmv_coverage
    ,round(sum(if(month = '202607',jz_gmv, null)) /sum(if(month = '202607',sku_gmv, null)),8)   as mth07_jz_gmv_coverage
    ,round(sum(if(month = '202608',jz_gmv, null)) /sum(if(month = '202608',sku_gmv, null)),8)   as mth08_jz_gmv_coverage
    ,round(sum(if(month = '202609',jz_gmv, null)) /sum(if(month = '202609',sku_gmv, null)),8)   as mth09_jz_gmv_coverage
    ,round(sum(if(month = '202610',jz_gmv, null)) /sum(if(month = '202610',sku_gmv, null)),8)   as mth10_jz_gmv_coverage
    ,round(sum(if(month = '202611',jz_gmv, null)) /sum(if(month = '202611',sku_gmv, null)),8)   as mth11_jz_gmv_coverage
    ,round(sum(if(month = '202612',jz_gmv, null)) /sum(if(month = '202612',sku_gmv, null)),8)   as mth12_jz_gmv_coverage
    ,round(sum(jz_gmv)/sum(sku_gmv),8)                                                          as ytd_jz_gmv_coverage
    ,''                                                                                         as interval_2
    ,round(sum(if(month = '202601',ul_sku_gmv,null)) /sum(if(month = '202601',sku_gmv, null)),8) as mth01_ul_sku_gmv_coverage
    ,round(sum(if(month = '202602',ul_sku_gmv,null)) /sum(if(month = '202602',sku_gmv, null)),8) as mth02_ul_sku_gmv_coverage
    ,round(sum(if(month = '202603',ul_sku_gmv,null)) /sum(if(month = '202603',sku_gmv, null)),8) as mth03_ul_sku_gmv_coverage
    ,round(sum(if(month = '202604',ul_sku_gmv,null)) /sum(if(month = '202604',sku_gmv, null)),8) as mth04_ul_sku_gmv_coverage
    ,round(sum(if(month = '202605',ul_sku_gmv,null)) /sum(if(month = '202605',sku_gmv, null)),8) as mth05_ul_sku_gmv_coverage
    ,round(sum(if(month = '202606',ul_sku_gmv,null)) /sum(if(month = '202606',sku_gmv, null)),8) as mth06_ul_sku_gmv_coverage
    ,round(sum(if(month = '202607',ul_sku_gmv,null)) /sum(if(month = '202607',sku_gmv, null)),8) as mth07_ul_sku_gmv_coverage
    ,round(sum(if(month = '202608',ul_sku_gmv,null)) /sum(if(month = '202608',sku_gmv, null)),8) as mth08_ul_sku_gmv_coverage
    ,round(sum(if(month = '202609',ul_sku_gmv,null)) /sum(if(month = '202609',sku_gmv, null)),8) as mth09_ul_sku_gmv_coverage
    ,round(sum(if(month = '202610',ul_sku_gmv,null)) /sum(if(month = '202610',sku_gmv, null)),8) as mth10_ul_sku_gmv_coverage
    ,round(sum(if(month = '202611',ul_sku_gmv,null)) /sum(if(month = '202611',sku_gmv, null)),8) as mth11_ul_sku_gmv_coverage
    ,round(sum(if(month = '202612',ul_sku_gmv,null)) /sum(if(month = '202612',sku_gmv, null)),8) as mth12_ul_sku_gmv_coverage
    ,round(sum(ul_sku_gmv)/sum(sku_gmv),8)                                                       as ytd_ul_sku_gmv_coverage
from ${spark.var.catalog.tmp}.fin.tmp_secondary_gmv_check
where 1=1 
and left(month,4)='2026'
group by store_name,bu
order by store_name,bu

store_name,bu,mth01_both_gmv_coverage,mth02_both_gmv_coverage,mth03_both_gmv_coverage,mth04_both_gmv_coverage,mth05_both_gmv_coverage,mth06_both_gmv_coverage,mth07_both_gmv_coverage,mth08_both_gmv_coverage,mth09_both_gmv_coverage,mth10_both_gmv_coverage,mth11_both_gmv_coverage,mth12_both_gmv_coverage,ytd_both_gmv_coverage,interval_1,mth01_jz_gmv_coverage,mth02_jz_gmv_coverage,mth03_jz_gmv_coverage,mth04_jz_gmv_coverage,mth05_jz_gmv_coverage,mth06_jz_gmv_coverage,mth07_jz_gmv_coverage,mth08_jz_gmv_coverage,mth09_jz_gmv_coverage,mth10_jz_gmv_coverage,mth11_jz_gmv_coverage,mth12_jz_gmv_coverage,ytd_jz_gmv_coverage,interval_2,mth01_ul_sku_gmv_coverage,mth02_ul_sku_gmv_coverage,mth03_ul_sku_gmv_coverage,mth04_ul_sku_gmv_coverage,mth05_ul_sku_gmv_coverage,mth06_ul_sku_gmv_coverage,mth07_ul_sku_gmv_coverage,mth08_ul_sku_gmv_coverage,mth09_ul_sku_gmv_coverage,mth10_ul_sku_gmv_coverage,mth11_ul_sku_gmv_coverage,mth12_ul_sku_gmv_coverage,ytd_ul_sku_gmv_coverage
DEWU - AHC,BEAUTY&WELLBEING,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0
JD - AHC,BEAUTY&WELLBEING,0.85701663,0.84252241,0.85538989,0.85529905,0.88113105,0.90764654,0.8970771,0.91070864,0.89357756,0.89263574,null,null,0.88010569,,0.86601967,0.8547333,0.86904694,0.86672664,0.88784759,0.90881741,0.89824524,0.9108534,0.89357756,0.89263574,null,null,0.88543339,,0.99099697,0.98778911,0.98634296,0.98857242,0.99328346,0.99882913,0.99883186,0.99985524,1.0,1.0,null,null,0.9946723
JD - AHC CBE,BEAUTY&WELLBEING,0.96283756,0.95524444,0.96416744,0.96178367,0.95822095,0.97331368,0.97671987,0.97682771,0.97533242,0.97838922,null,null,0.9671851,,0.96283756,0.95524444,0.96416744,0.96178367,0.95822095,0.97331368,0.97671987,0.97682771,0.97533242,0.97838922,null,null,0.9671851,,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,null,null,1.0
JD - UL,BEAUTY&WELLBEING,0.90497837,0.8972425,0.89938122,0.90397356,0.90223603,0.90763774,0.89855569,0.89359942,0.88222127,0.87775708,null,null,0.89668052,,0.90497837,0.8972425,0.89938122,0.90397356,0.90223603,0.90763774,0.89855569,0.89359942,0.88222127,0.87775708,null,null,0.89668052,,1.0,1.0,1.0,0.99957478,0.99800514,0.99820467,0.99689408,0.9951506,0.9923007,0.98962763,null,null,0.99668701
JD - UL,HOME CARE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,null,null,1.0
JD - UL,PERSONAL CARE,0.92686941,0.90845449,0.9329445,0.93429562,0.94414818,0.93213789,0.92796995,0.93603208,0.92789014,0.92909283,null,null,0.93130339,,0.92686941,0.90845449,0.9329445,0.93429562,0.94414818,0.93213789,0.92796995,0.93603208,0.92789014,0.92909283,null,null,0.93130339,,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,null,null,1.0
PDD - AHC,BEAUTY&WELLBEING,null,null,null,null,null,null,null,null,null,null,null,null,null,,null,null,null,null,null,null,null,null,null,null,null,null,null,,null,null,null,null,null,null,null,null,null,null,null,null,null
PDD - AHC DTC,BEAUTY&WELLBEING,null,null,null,null,null,null,null,null,null,null,null,null,null,,null,null,null,null,null,null,null,null,null,null,null,null,null,,null,null,null,null,null,null,null,null,null,null,null,null,null
PDD - UL,BEAUTY&WELLBEING,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,0.89733079,0.88513378,0.88121376,0.87785718,0.88879262,0.89120265,0.89196084,0.896228,0.88835841,0.8661032,null,null,0.88624726,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0
PDD - UL,HOME CARE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0


In [0]:
-- 二级gmv gap, By month+store+bu+category
select
    store_name,bu,category_name
    ,round(sum(if(month = '202601',both_gmv, null)) /sum(if(month = '202601',sku_gmv, null)),8) as mth01_both_gmv_coverage
    ,round(sum(if(month = '202602',both_gmv, null)) /sum(if(month = '202602',sku_gmv, null)),8) as mth02_both_gmv_coverage
    ,round(sum(if(month = '202603',both_gmv, null)) /sum(if(month = '202603',sku_gmv, null)),8) as mth03_both_gmv_coverage
    ,round(sum(if(month = '202604',both_gmv, null)) /sum(if(month = '202604',sku_gmv, null)),8) as mth04_both_gmv_coverage
    ,round(sum(if(month = '202605',both_gmv, null)) /sum(if(month = '202605',sku_gmv, null)),8) as mth05_both_gmv_coverage
    ,round(sum(if(month = '202606',both_gmv, null)) /sum(if(month = '202606',sku_gmv, null)),8) as mth06_both_gmv_coverage
    ,round(sum(if(month = '202607',both_gmv, null)) /sum(if(month = '202607',sku_gmv, null)),8) as mth07_both_gmv_coverage
    ,round(sum(if(month = '202608',both_gmv, null)) /sum(if(month = '202608',sku_gmv, null)),8) as mth08_both_gmv_coverage
    ,round(sum(if(month = '202609',both_gmv, null)) /sum(if(month = '202609',sku_gmv, null)),8) as mth09_both_gmv_coverage
    ,round(sum(if(month = '202610',both_gmv, null)) /sum(if(month = '202610',sku_gmv, null)),8) as mth10_both_gmv_coverage
    ,round(sum(if(month = '202611',both_gmv, null)) /sum(if(month = '202611',sku_gmv, null)),8) as mth11_both_gmv_coverage
    ,round(sum(if(month = '202612',both_gmv, null)) /sum(if(month = '202612',sku_gmv, null)),8) as mth12_both_gmv_coverage
    ,round(sum(both_gmv)/sum(sku_gmv),8)                                                        as ytd_both_gmv_coverage
    ,''                                                                                         as interval_1
    ,round(sum(if(month = '202601',jz_gmv, null)) /sum(if(month = '202601',sku_gmv, null)),8)   as mth01_jz_gmv_coverage
    ,round(sum(if(month = '202602',jz_gmv, null)) /sum(if(month = '202602',sku_gmv, null)),8)   as mth02_jz_gmv_coverage
    ,round(sum(if(month = '202603',jz_gmv, null)) /sum(if(month = '202603',sku_gmv, null)),8)   as mth03_jz_gmv_coverage
    ,round(sum(if(month = '202604',jz_gmv, null)) /sum(if(month = '202604',sku_gmv, null)),8)   as mth04_jz_gmv_coverage
    ,round(sum(if(month = '202605',jz_gmv, null)) /sum(if(month = '202605',sku_gmv, null)),8)   as mth05_jz_gmv_coverage
    ,round(sum(if(month = '202606',jz_gmv, null)) /sum(if(month = '202606',sku_gmv, null)),8)   as mth06_jz_gmv_coverage
    ,round(sum(if(month = '202607',jz_gmv, null)) /sum(if(month = '202607',sku_gmv, null)),8)   as mth07_jz_gmv_coverage
    ,round(sum(if(month = '202608',jz_gmv, null)) /sum(if(month = '202608',sku_gmv, null)),8)   as mth08_jz_gmv_coverage
    ,round(sum(if(month = '202609',jz_gmv, null)) /sum(if(month = '202609',sku_gmv, null)),8)   as mth09_jz_gmv_coverage
    ,round(sum(if(month = '202610',jz_gmv, null)) /sum(if(month = '202610',sku_gmv, null)),8)   as mth10_jz_gmv_coverage
    ,round(sum(if(month = '202611',jz_gmv, null)) /sum(if(month = '202611',sku_gmv, null)),8)   as mth11_jz_gmv_coverage
    ,round(sum(if(month = '202612',jz_gmv, null)) /sum(if(month = '202612',sku_gmv, null)),8)   as mth12_jz_gmv_coverage
    ,round(sum(jz_gmv)/sum(sku_gmv),8)                                                          as ytd_jz_gmv_coverage
    ,''                                                                                         as interval_2
    ,round(sum(if(month = '202601',ul_sku_gmv,null)) /sum(if(month = '202601',sku_gmv, null)),8) as mth01_ul_sku_gmv_coverage
    ,round(sum(if(month = '202602',ul_sku_gmv,null)) /sum(if(month = '202602',sku_gmv, null)),8) as mth02_ul_sku_gmv_coverage
    ,round(sum(if(month = '202603',ul_sku_gmv,null)) /sum(if(month = '202603',sku_gmv, null)),8) as mth03_ul_sku_gmv_coverage
    ,round(sum(if(month = '202604',ul_sku_gmv,null)) /sum(if(month = '202604',sku_gmv, null)),8) as mth04_ul_sku_gmv_coverage
    ,round(sum(if(month = '202605',ul_sku_gmv,null)) /sum(if(month = '202605',sku_gmv, null)),8) as mth05_ul_sku_gmv_coverage
    ,round(sum(if(month = '202606',ul_sku_gmv,null)) /sum(if(month = '202606',sku_gmv, null)),8) as mth06_ul_sku_gmv_coverage
    ,round(sum(if(month = '202607',ul_sku_gmv,null)) /sum(if(month = '202607',sku_gmv, null)),8) as mth07_ul_sku_gmv_coverage
    ,round(sum(if(month = '202608',ul_sku_gmv,null)) /sum(if(month = '202608',sku_gmv, null)),8) as mth08_ul_sku_gmv_coverage
    ,round(sum(if(month = '202609',ul_sku_gmv,null)) /sum(if(month = '202609',sku_gmv, null)),8) as mth09_ul_sku_gmv_coverage
    ,round(sum(if(month = '202610',ul_sku_gmv,null)) /sum(if(month = '202610',sku_gmv, null)),8) as mth10_ul_sku_gmv_coverage
    ,round(sum(if(month = '202611',ul_sku_gmv,null)) /sum(if(month = '202611',sku_gmv, null)),8) as mth11_ul_sku_gmv_coverage
    ,round(sum(if(month = '202612',ul_sku_gmv,null)) /sum(if(month = '202612',sku_gmv, null)),8) as mth12_ul_sku_gmv_coverage
    ,round(sum(ul_sku_gmv)/sum(sku_gmv),8)                                                       as ytd_ul_sku_gmv_coverage
from ${spark.var.catalog.tmp}.fin.tmp_secondary_gmv_check
where 1=1 
and left(month,4)='2026'
group by store_name,bu,category_name
order by store_name,bu,category_name

store_name,bu,category_name,mth01_both_gmv_coverage,mth02_both_gmv_coverage,mth03_both_gmv_coverage,mth04_both_gmv_coverage,mth05_both_gmv_coverage,mth06_both_gmv_coverage,mth07_both_gmv_coverage,mth08_both_gmv_coverage,mth09_both_gmv_coverage,mth10_both_gmv_coverage,mth11_both_gmv_coverage,mth12_both_gmv_coverage,ytd_both_gmv_coverage,interval_1,mth01_jz_gmv_coverage,mth02_jz_gmv_coverage,mth03_jz_gmv_coverage,mth04_jz_gmv_coverage,mth05_jz_gmv_coverage,mth06_jz_gmv_coverage,mth07_jz_gmv_coverage,mth08_jz_gmv_coverage,mth09_jz_gmv_coverage,mth10_jz_gmv_coverage,mth11_jz_gmv_coverage,mth12_jz_gmv_coverage,ytd_jz_gmv_coverage,interval_2,mth01_ul_sku_gmv_coverage,mth02_ul_sku_gmv_coverage,mth03_ul_sku_gmv_coverage,mth04_ul_sku_gmv_coverage,mth05_ul_sku_gmv_coverage,mth06_ul_sku_gmv_coverage,mth07_ul_sku_gmv_coverage,mth08_ul_sku_gmv_coverage,mth09_ul_sku_gmv_coverage,mth10_ul_sku_gmv_coverage,mth11_ul_sku_gmv_coverage,mth12_ul_sku_gmv_coverage,ytd_ul_sku_gmv_coverage
DEWU - AHC,BEAUTY&WELLBEING,SKIN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0
JD - AHC,BEAUTY&WELLBEING,SKIN,0.85701663,0.84252241,0.85538989,0.85529905,0.88113105,0.90764654,0.8970771,0.91070864,0.89357756,0.89263574,null,null,0.88010569,,0.86601967,0.8547333,0.86904694,0.86672664,0.88784759,0.90881741,0.89824524,0.9108534,0.89357756,0.89263574,null,null,0.88543339,,0.99099697,0.98778911,0.98634296,0.98857242,0.99328346,0.99882913,0.99883186,0.99985524,1.0,1.0,null,null,0.9946723
JD - AHC CBE,BEAUTY&WELLBEING,SKIN,0.96283756,0.95524444,0.96416744,0.96178367,0.95822095,0.97331368,0.97671987,0.97682771,0.97533242,0.97838922,null,null,0.9671851,,0.96283756,0.95524444,0.96416744,0.96178367,0.95822095,0.97331368,0.97671987,0.97682771,0.97533242,0.97838922,null,null,0.9671851,,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,null,null,1.0
JD - UL,BEAUTY&WELLBEING,HAIR,0.94915134,0.93570499,0.9313598,0.9195777,0.92230112,0.91784477,0.91878973,0.90852968,0.89557225,0.8945949,null,null,0.91766395,,0.94915134,0.93570499,0.9313598,0.9195777,0.92230112,0.91784477,0.91878973,0.90852968,0.89557225,0.8945949,null,null,0.91766395,,1.0,1.0,1.0,0.99945221,0.99749247,0.99795122,0.99618527,0.99446846,0.99089896,0.98723763,null,null,0.99605034
JD - UL,BEAUTY&WELLBEING,SKIN,0.81569122,0.81915522,0.82693074,0.84984359,0.82415946,0.83533719,0.81128155,0.83584928,0.84100597,0.83795287,null,null,0.82970741,,0.81569122,0.81915522,0.82693074,0.84984359,0.82415946,0.83533719,0.81128155,0.83584928,0.84100597,0.83795287,null,null,0.82970741,,1.0,1.0,1.0,1.0,1.0,1.0,0.99995135,0.99778912,0.99662795,0.99527754,null,null,0.9987191
JD - UL,HOME CARE,FCL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,null,null,1.0
JD - UL,HOME CARE,FEN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,null,null,1.0
JD - UL,HOME CARE,HHC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,null,null,1.0
JD - UL,PERSONAL CARE,DEO,0.9999026,0.99996971,1.0,1.0,0.99997153,0.99999177,1.0,1.0,0.99548134,0.87874553,null,null,0.98802351,,0.9999026,0.99996971,1.0,1.0,0.99997153,0.99999177,1.0,1.0,0.99548134,0.87874553,null,null,0.98802351,,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,null,null,1.0
JD - UL,PERSONAL CARE,ORAL,0.99996509,0.99999513,0.99934582,0.9995821,0.99685805,0.98694251,0.96745513,0.94956763,0.85437346,0.87186858,null,null,0.9502489,,0.99996509,0.99999513,0.99934582,0.9995821,0.99685805,0.98694251,0.96745513,0.94956763,0.85437346,0.87186858,null,null,0.9502489,,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,null,null,1.0


In [0]:
-- item vs sku gap, By store+bu+category+brand
select   
    store_name,bu,category_name,brand_name
    ,round(sum(if(month = '202601', item_sku_gap, null))) as mth01_gap_absolute
    ,round(sum(if(month = '202602', item_sku_gap, null))) as mth02_gap_absolute
    ,round(sum(if(month = '202603', item_sku_gap, null))) as mth03_gap_absolute
    ,round(sum(if(month = '202604', item_sku_gap, null))) as mth04_gap_absolute
    ,round(sum(if(month = '202605', item_sku_gap, null))) as mth05_gap_absolute
    ,round(sum(if(month = '202606', item_sku_gap, null))) as mth06_gap_absolute
    ,round(sum(if(month = '202607', item_sku_gap, null))) as mth07_gap_absolute
    ,round(sum(if(month = '202608', item_sku_gap, null))) as mth08_gap_absolute
    ,round(sum(if(month = '202609', item_sku_gap, null))) as mth09_gap_absolute
    ,round(sum(if(month = '202610', item_sku_gap, null))) as mth10_gap_absolute
    ,round(sum(if(month = '202611', item_sku_gap, null))) as mth11_gap_absolute
    ,round(sum(if(month = '202612', item_sku_gap, null))) as mth12_gap_absolute
    ,round(sum(item_sku_gap))                             as ytd_absolute
    ,round(mth01_gap_absolute /sum(if(month='202601',if(store_name like 'TMALL FGS%',item_gmv,item_gmv_after_refund),null)),8) as mth01_gap_rate
    ,round(mth02_gap_absolute /sum(if(month='202602',if(store_name like 'TMALL FGS%',item_gmv,item_gmv_after_refund),null)),8) as mth02_gap_rate
    ,round(mth03_gap_absolute /sum(if(month='202603',if(store_name like 'TMALL FGS%',item_gmv,item_gmv_after_refund),null)),8) as mth03_gap_rate
    ,round(mth04_gap_absolute /sum(if(month='202604',if(store_name like 'TMALL FGS%',item_gmv,item_gmv_after_refund),null)),8) as mth04_gap_rate
    ,round(mth05_gap_absolute /sum(if(month='202605',if(store_name like 'TMALL FGS%',item_gmv,item_gmv_after_refund),null)),8) as mth05_gap_rate
    ,round(mth06_gap_absolute /sum(if(month='202606',if(store_name like 'TMALL FGS%',item_gmv,item_gmv_after_refund),null)),8) as mth06_gap_rate
    ,round(mth07_gap_absolute /sum(if(month='202607',if(store_name like 'TMALL FGS%',item_gmv,item_gmv_after_refund),null)),8) as mth07_gap_rate
    ,round(mth08_gap_absolute /sum(if(month='202608',if(store_name like 'TMALL FGS%',item_gmv,item_gmv_after_refund),null)),8) as mth08_gap_rate
    ,round(mth09_gap_absolute /sum(if(month='202609',if(store_name like 'TMALL FGS%',item_gmv,item_gmv_after_refund),null)),8) as mth09_gap_rate
    ,round(mth10_gap_absolute /sum(if(month='202610',if(store_name like 'TMALL FGS%',item_gmv,item_gmv_after_refund),null)),8) as mth10_gap_rate
    ,round(mth11_gap_absolute /sum(if(month='202611',if(store_name like 'TMALL FGS%',item_gmv,item_gmv_after_refund),null)),8) as mth11_gap_rate
    ,round(mth12_gap_absolute /sum(if(month='202612',if(store_name like 'TMALL FGS%',item_gmv,item_gmv_after_refund),null)),8) as mth12_gap_rate
    ,round(ytd_absolute /sum(if(store_name like 'TMALL FGS%',item_gmv,item_gmv_after_refund)),8) as ytd_gap_rate
from ${spark.var.catalog.tmp}.fin.tmp_secondary_gmv_check
where  left(month,4) ='2026'
group by store_name,bu,category_name,brand_name
order by store_name,bu,category_name,brand_name

store_name,bu,category_name,brand_name,mth01_gap_absolute,mth02_gap_absolute,mth03_gap_absolute,mth04_gap_absolute,mth05_gap_absolute,mth06_gap_absolute,mth07_gap_absolute,mth08_gap_absolute,mth09_gap_absolute,mth10_gap_absolute,mth11_gap_absolute,mth12_gap_absolute,ytd_absolute,mth01_gap_rate,mth02_gap_rate,mth03_gap_rate,mth04_gap_rate,mth05_gap_rate,mth06_gap_rate,mth07_gap_rate,mth08_gap_rate,mth09_gap_rate,mth10_gap_rate,mth11_gap_rate,mth12_gap_rate,ytd_gap_rate
DEWU - AHC,BEAUTY&WELLBEING,SKIN,AHC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0
JD - AHC,BEAUTY&WELLBEING,SKIN,AHC,0.0,0.0,0.0,1.0,1.0,0.0,0.0,-1.0,0.0,0.0,null,null,1.0,0.0,0.0,0.0,2.8E-7,1.8E-7,0.0,0.0,-3.0E-7,0.0,0.0,null,null,2.0E-8
JD - AHC CBE,BEAUTY&WELLBEING,SKIN,AHC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0
JD - UL,BEAUTY&WELLBEING,HAIR,CLEAR,0.0,0.0,-1.0,0.0,-1.0,-2.1801703E7,0.0,0.0,-1.0,674.0,null,null,-2.1801032E7,0.0,0.0,-6.0E-8,0.0,-4.0E-8,-0.71612008,0.0,0.0,-4.0E-8,2.22E-5,null,null,-0.096037
JD - UL,BEAUTY&WELLBEING,HAIR,DOVE,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,null,null,2.0,2.8E-7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.5E-7,null,null,5.0E-8
JD - UL,BEAUTY&WELLBEING,HAIR,HAZELINE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0
JD - UL,BEAUTY&WELLBEING,HAIR,LUX,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0
JD - UL,BEAUTY&WELLBEING,HAIR,NEXXUS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,977.0,3247.0,null,null,4224.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00442067,0.01167767,null,null,0.0012939
JD - UL,BEAUTY&WELLBEING,SKIN,DOVE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0
JD - UL,BEAUTY&WELLBEING,SKIN,PONDS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,null,0.0


In [0]:
-- 二级gmv gap, raw data
select * 
from ${spark.var.catalog.tmp}.fin.tmp_secondary_gmv_check
where  left(month,4) ='2026'
order by month,store_name,bu,category_name,brand_name,product_line_group

month,store_name,bu,category_name,brand_name,product_line_group,item_gmv_after_refund,item_gmv,sku_gmv,both_gmv,jz_gmv,ul_sku_gmv,item_sku_gap
202601,DEWU - AHC,BEAUTY&WELLBEING,SKIN,AHC,null,0.00,0.00,94807.0,0.0,0.0,0.0,-94807.0
202601,DEWU - AHC,BEAUTY&WELLBEING,SKIN,AHC,B5,62277.00,62277.00,0.0,0.0,0.0,0.0,62277.0
202601,DEWU - AHC,BEAUTY&WELLBEING,SKIN,AHC,HA,2823.00,2823.00,0.0,0.0,0.0,0.0,2823.0
202601,DEWU - AHC,BEAUTY&WELLBEING,SKIN,AHC,LE SALON,16664.00,16664.00,0.0,0.0,0.0,0.0,16664.0
202601,DEWU - AHC,BEAUTY&WELLBEING,SKIN,AHC,MAN,3549.00,3549.00,0.0,0.0,0.0,0.0,3549.0
202601,DEWU - AHC,BEAUTY&WELLBEING,SKIN,AHC,MASK - B5,2705.00,2705.00,0.0,0.0,0.0,0.0,2705.0
202601,DEWU - AHC,BEAUTY&WELLBEING,SKIN,AHC,OTHERS,5107.00,5107.00,0.0,0.0,0.0,0.0,5107.0
202601,DEWU - AHC,BEAUTY&WELLBEING,SKIN,AHC,SUN,1682.00,1682.00,0.0,0.0,0.0,0.0,1682.0
202601,JD - AHC,BEAUTY&WELLBEING,SKIN,AHC,null,0.00,0.00,38406.0,0.0,38812.0,-406.0,-38406.0
202601,JD - AHC,BEAUTY&WELLBEING,SKIN,AHC,B5,1747345.00,1747345.00,1688615.0,1618182.0,1618182.0,1688615.0,58730.0


In [0]:
%sql

  select
    month
    ,store_name
    ,category_name
    ,brand_name
    ,product_line_group
    ,secondary_after_gmv_amount as gmv_after_refund
    ,secondary_gmv_amount       as gmv
    ,bu_name
  from ${spark.var.catalog.ads}.fin_cpt.ads_second_total_sales_mf
  where dt = '$dt' and abs(secondary_gmv_amount) > 0
  and month='202601'
  and sales_type='2C'
  and store_name='PDD - VSL'
  and category_name='SKIN'
  and brand_name='VASELINE'

month,store_name,category_name,brand_name,product_line_group,gmv_after_refund,gmv,bu_name
202601,PDD - VSL,SKIN,VASELINE,CORE,539802.460000,539802.460000,BEAUTY&WELLBEING
202601,PDD - VSL,SKIN,VASELINE,DERMA,198078.380000,198078.380000,BEAUTY&WELLBEING
202601,PDD - VSL,SKIN,VASELINE,LIP&CUTTIE,1787161.820000,1787161.820000,BEAUTY&WELLBEING
202601,PDD - VSL,SKIN,VASELINE,PJ,1070396.040000,1070396.040000,BEAUTY&WELLBEING
202601,PDD - VSL,SKIN,VASELINE,HC,1201028.840000,1201028.840000,BEAUTY&WELLBEING
202601,PDD - VSL,SKIN,VASELINE,GH,16860.060000,16860.060000,BEAUTY&WELLBEING
202601,PDD - VSL,SKIN,VASELINE,OTHERS,235894.540000,235894.540000,BEAUTY&WELLBEING
202601,PDD - VSL,SKIN,VASELINE,IR,13361.100000,13361.100000,BEAUTY&WELLBEING


In [0]:
    SELECT
        REPLACE(LEFT(period_key, 7), '-', '') AS month,
        customer AS store_name,
        bu,
        category AS category_name,
        brand AS brand_name,
        product_line_group,
        SUM(gmv) AS gmv
    FROM (
        SELECT 
            period_key,
            customer,
            bu,
            category,
            brand,
            product_line_group,
            gmv
        FROM ${spark.var.catalog.tmp}.fin.tmp_trd_total_item_sales_all
        WHERE ABS(gmv) > 0
    ) t
    WHERE REPLACE(LEFT(period_key, 7), '-', '') = '202601'
        AND customer = 'TMALL SUPER - UL'
        AND bu = 'BEAUTY&WELLBEING'
        AND category = 'SKIN'
        AND brand = 'AHC'
    GROUP BY 
        LEFT(period_key, 7),
        customer,
        bu,
        category,
        brand,
        product_line_group

month,store_name,bu,category_name,brand_name,product_line_group,gmv
202601,TMALL SUPER - UL,BEAUTY&WELLBEING,SKIN,AHC,B5,44578.259999999995
202601,TMALL SUPER - UL,BEAUTY&WELLBEING,SKIN,AHC,HA,130.0
202601,TMALL SUPER - UL,BEAUTY&WELLBEING,SKIN,AHC,SUN,2626.3199999999997
202601,TMALL SUPER - UL,BEAUTY&WELLBEING,SKIN,AHC,MAN,11962.92
202601,TMALL SUPER - UL,BEAUTY&WELLBEING,SKIN,AHC,MASK - OTHERS,788.0
202601,TMALL SUPER - UL,BEAUTY&WELLBEING,SKIN,AHC,MASK - B5 RENO,9942.059999999998
202601,TMALL SUPER - UL,BEAUTY&WELLBEING,SKIN,AHC,LE SALON,5370.480000000001
202601,TMALL SUPER - UL,BEAUTY&WELLBEING,SKIN,AHC,null,22774.8


In [0]:
%sql

select 
    replace(substr(t1.period_key,1,7),'-','')     as month
    ,t3.bu                                        as bu_name
    -- ,t2.customer_type                             as customer_type
    -- ,t2.customer_group                            as site_name
    -- ,if(t1.period_key > '2025-10-16' and t1.shop_id = '621699362', 'PDD - VSL', t2.customer) as store_name    -- 20251204修改
    ,t3.category                                  as category_name
    ,t3.brand                                     as brand_name
    ,t3.product_line_group                        as product_line_group
    ,t1.gmv_after_refund                          as secondary_after_gmv_amount
    -- ,sum(coalesce(gmv_after_refund,0))             as secondary_after_gmv_amount
    ,t1.*
from ${spark.var.catalog.dws}.ddm.dws_trd_total_item_sales_df t1
left join (
  select *
  from ${spark.var.catalog.dim}.ddm.dim_prd_item
  where is_current = 1 
) t3
on t1.item_id = t3.item_id and if(t1.platform in ('VIP','TMALL SUPER'),t1.shop_id,t1.platform) = if(t3.platform in ('VIP','TMALL SUPER'),t3.shop_id,t3.platform)
and t1.period_key between t3.start_time and t3.end_time
where substr(t1.period_key,1,7)='2026-03'
and t1.shop_id='621699362'
and t3.bu ='BEAUTY&WELLBEING'   
and t3.category  ='SKIN'
and t3.brand ='VASELINE'
-- group by replace(substr(t1.period_key,1,7),'-',''),t3.bu,t3.category  ,t3.brand,t3.product_line_group
--  limit 10 

month,bu_name,category_name,brand_name,product_line_group,secondary_after_gmv_amount,source_shop_id,source_shop_name,shop_id,shop_name,ulstore_code,ulstore_name,source_type,item_id,item_name,reference_sku_id,platform_cate_level1_id,platform_cate_level2_id,platform_cate_level3_id,platform_cate_level1_name,platform_cate_level2_name,platform_cate_level3_name,period_key,period_value,start_date,end_date,gmv,refund_gmv,gmv_after_refund,qty,order_cnt,refund_order_cnt,buyer_cnt,add_cart_qty,add_cart_buyer_cnt,favorite_buyer_cnt,pv,uv,imp_pv,imp_uv,avg_stay_time,bounce_rate,cps_gmv,cps_order_cnt,cps_pv,cps_uv,cps_buyer_cnt,cps_qty,cps_gmv_after_refund,pfm_imp,pfm_click,pfm_spending,pfm_direct_order_cnt,pfm_direct_gmv,pfm_order_cnt,pfm_gmv,self_live_gmv,self_live_order_cnt,self_live_pv,self_live_uv,self_live_buyer_cnt,self_live_qty,self_live_gmv_after_refund,coop_live_gmv,coop_live_order_cnt,coop_live_pv,coop_live_uv,coop_live_buyer_cnt,coop_live_qty,coop_live_gmv_after_refund,bulk_qty,bulk_gmv,etl_time,platform,period_type
202603,BEAUTY&WELLBEING,SKIN,VASELINE,HC,631.20,621699362,Vaseline凡士林官方旗舰店,621699362,Vaseline凡士林官方旗舰店,20251009EC00001,VASELINE凡士林官方旗舰店,DTC,528298885827,凡士林护手霜清香滋润保湿温和补水润手霜护甲不油腻男女学生锁水,1852680134739,null,null,null,采集,null,null,2026-03-01,20260301,2026-03-01,2026-03-01,631.20,null,631.20,null,27.0,null,27.0,null,null,null,301,301,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2026-11-10 16:42:02.392716,PDD,日
202603,BEAUTY&WELLBEING,SKIN,VASELINE,PJ,268.90,621699362,Vaseline凡士林官方旗舰店,621699362,Vaseline凡士林官方旗舰店,20251009EC00001,VASELINE凡士林官方旗舰店,DTC,512644540862,凡士林晶冻保湿补水身体乳手足护理有效防干裂滋润身体男女通用,1852684789854,null,null,null,采集,null,null,2026-03-02,20260302,2026-03-02,2026-03-02,268.90,null,268.90,null,8.0,null,8.0,null,null,null,107,107,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2026-11-10 16:42:02.392716,PDD,日
202603,BEAUTY&WELLBEING,SKIN,VASELINE,DERMA,0.00,621699362,Vaseline凡士林官方旗舰店,621699362,Vaseline凡士林官方旗舰店,20251009EC00001,VASELINE凡士林官方旗舰店,DTC,660404616238,凡士林大白罐雪融霜修护屏障敏肌全身可用修护肌肤屏障大容量官方,null,null,null,null,采集,null,null,2026-03-03,20260303,2026-03-03,2026-03-03,0.00,null,0.00,null,null,null,null,null,null,null,13,13,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2026-11-10 16:42:02.392716,PDD,日
202603,BEAUTY&WELLBEING,SKIN,VASELINE,GH,237.70,621699362,Vaseline凡士林官方旗舰店,621699362,Vaseline凡士林官方旗舰店,20251009EC00001,VASELINE凡士林官方旗舰店,DTC,447068475590,凡士林身体露修美白焕亮滋润保湿清爽不黏腻舒缓有效补水官方,1862706844302,null,null,null,采集,null,null,2026-03-04,20260304,2026-03-04,2026-03-04,237.70,null,237.70,null,5.0,null,5.0,null,null,null,170,170,null,null,null,null,null,null,null,null,null,null,null,813.00,84.00,54.28,null,192.6,5.00,250.50,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2026-11-10 16:42:02.392716,PDD,日
202603,BEAUTY&WELLBEING,SKIN,VASELINE,HC,457.09,621699362,Vaseline凡士林官方旗舰店,621699362,Vaseline凡士林官方旗舰店,20251009EC00001,VASELINE凡士林官方旗舰店,DTC,528298885827,凡士林护手霜清香滋润保湿温和补水润手霜护甲不油腻男女学生锁水,1852680134740,null,null,null,采集,null,null,2026-03-04,20260304,2026-03-04,2026-03-04,457.09,null,457.09,null,20.0,null,20.0,null,null,null,283,283,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2026-11-10 16:42:02.392716,PDD,日
202603,BEAUTY&WELLBEING,SKIN,VASELINE,LIP&CUTTIE,0.00,621699362,Vaseline凡士林官方旗舰店,621699362,Vaseline凡士林官方旗舰店,20251009EC00001,VASELINE凡士林官方旗舰店,DTC,676658893060,凡士林润唇膏滋润保湿修护干燥薄荷润唇膏清爽不黏腻润泽无色保湿,null,null,null,null,采集,null,null,2026-03-06,20260306,2026-03-06,2026-03-06,0.00,null,0.00,null,null,null,null,null,null,null,0,0,null,n

In [0]:
%sql

-- 数据合并（方便对数）

with cte_data as 
(
  select 
    period_key,shop_id,item_id,sku_id,customer,sku_code,gmv_before_refund,gmv,pay_qty,bom_num,is_bundle
    ,listprice,total_listprice,total_bom_num,lp_contri,source_type,platform,bu_name,category_name,brand_name,product_line_group
  from ${spark.var.catalog.tmp}.fin.tmp_trd_total_item_sales_vip_bundle
  union all
  select  
    period_key,shop_id,item_id,sku_id,customer,sku_code,gmv_before_refund,gmv,pay_qty,bom_num,is_bundle
    ,listprice,total_listprice,total_bom_num,lp_contri,source_type,platform,bu_name,category_name,brand_name,product_line_group
  from ${spark.var.catalog.tmp}.fin.tmp_trd_total_item_sales_tm_bundle
  union all
  select
    period_key,shop_id,item_id,sku_id,customer,sku_code,new_gmv_before_refund as gmv_before_refund,new_gmv as gmv,pay_qty,bom_num,is_bundle
    ,listprice,total_listprice,total_bom_num,lp_contri,source_type,platform,bu_name,category_name,brand_name,product_line_group
  from ${spark.var.catalog.tmp}.fin.tmp_trd_total_item_sales_bundle
) ,
source_data as (
select
  a.*
  ,nvl(a.pay_qty,0)* nvl(a.listprice,0)             as gsv
  ,case when a.platform = 'JD' then c.pack_size else b.pack_size end as pack_size
  ,case when a.platform = 'JD' then c.core_pack_group else b.core_pack_group end as core_pack_group
  ,case when a.platform = 'JD' then c.is_main_product else b.is_main_product end as is_main_product
  ,case when a.customer in ('TMALL FGS - AHC CBE','TMALL FGS - AHC') 
            and nvl(b.flag,0) = 1 and nvl(d.flag,0) = 1          then 1
        when a.platform = 'JD' and nvl(c.flag,0) = 1 and nvl(f.flag,0) = 1 then 1
        when  nvl(b.flag,0)+nvl(c.flag,0) >= 1 then 1 else 0 end as flag
  ,case when a.platform = 'JD' then f.item_name else d.item_name end as item_name
  ,case when a.platform = 'JD' then f.category else d.category end as category
  ,case when a.platform = 'JD' then f.brand else d.brand end as brand
  ,case when a.platform = 'JD' then f.bu else d.bu end as bu
  ,e.shop_name
  -- ,case when a.platform = 'JD' then f.product_line_group else d.product_line_group end as product_line_group
  ,e.customer_type
  ,e.site_name
from cte_data a 
left join 
(
  select 
    distinct item_id,sku_id,shop_id,max(pack_size) as pack_size
    ,core_pack_group,max(is_main_product) as is_main_product
    ,1  as flag
  from ${spark.var.catalog.dim}.ddm.dim_sku_pack_size
  where end_time = '9999-12-31' 
  and channel <> 'DOUYIN'
  group by item_id,sku_id,shop_id,core_pack_group
) b 
  on a.item_id = b.item_id and a.sku_id = b.sku_id and a.shop_id = b.shop_id and a.platform <> 'JD'
left join 
(
  select 
    distinct item_id,shop_id,max(pack_size) as pack_size
    ,core_pack_group,max(is_main_product) as is_main_product
    ,1 as flag
  from ${spark.var.catalog.dim}.ddm.dim_sku_pack_size
  where end_time = '9999-12-31' 
  and channel <> 'DOUYIN'
  group by item_id,shop_id,core_pack_group
) c
  on a.item_id = c.item_id and  a.shop_id = c.shop_id and a.platform = 'JD'
left join (
    select *,1 as flag
    from ${spark.var.catalog.dim}.ddm.dim_prd_sku   -- 打标表
    where is_current = 1
  ) d
  on a.item_id = d.item_id and a.sku_id = d.sku_id 
  and if(a.platform in ('VIP','TMALL SUPER'),a.shop_id,a.platform) = if(d.platform in ('VIP','TMALL SUPER'),d.shop_id,d.platform)
  and a.period_key between d.start_time and d.end_time and a.platform <> 'JD'
  and d.item_name not like '%购物金%' 
left join 
(
      select  distinct upper(trim(customer)) as customer,shop_id,shop_name
      ,customer_type,customer_group as site_name
    from ${spark.var.catalog.dim}.ddm.dim_store
    where status = 1  and shop_id <> ''
)e 
  on a.shop_id =e.shop_id and a.customer = e.customer
left join (
    select *,1 as flag
    from ${spark.var.catalog.dim}.ddm.dim_prd_sku   -- 打标表
    where is_current = 1
  ) f
  on a.item_id = f.item_id 
  and if(a.platform in ('VIP','TMALL SUPER'),a.shop_id,a.platform) = if(d.platform in ('VIP','TMALL SUPER'),d.shop_id,d.platform)
  and a.period_key between f.start_time and f.end_time and f.platform = 'JD'
  and f.item_name not like '%购物金%' 
)
select * from source_data
  where bu_name='BEAUTY&WELLBEING'
  and category_name='SKIN'
  and brand_name='AHC'
  and substr(period_key,1,7)='2026-01'
  and shop_id='725677994'

period_key,shop_id,item_id,sku_id,customer,sku_code,gmv_before_refund,gmv,pay_qty,bom_num,is_bundle,listprice,total_listprice,total_bom_num,lp_contri,source_type,platform,bu_name,category_name,brand_name,product_line_group,gsv,pack_size,core_pack_group,is_main_product,flag,item_name,category,brand,bu,shop_name,customer_type,site_name
2026-01-26,725677994,954266923310,5879446589873,TMALL SUPER - UL,64355185,554.44,371.22,2,0,0,123.0500000000000000000000,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,SKIN,AHC,MAN,246.1,null,null,null,0,爱和纯 男士水乳洁面三件套 保湿补水锁水控油清爽泡沫清洁 380ML,SKIN,AHC,BEAUTY&WELLBEING,天猫超市,KA EC,TMALL SUPER
2026-01-05,725677994,955374729026,6044320710880,TMALL SUPER - UL,69764365,77.51,77.51,1,0,0,79.2500000000000000000000,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,SKIN,AHC,SUN,79.25,null,null,null,0,【优惠价】A.H.C/爱和纯 臻致水合防晒霜,SKIN,AHC,BEAUTY&WELLBEING,天猫超市,KA EC,TMALL SUPER
2026-01-12,725677994,955381365598,5879430741936,TMALL SUPER - UL,64950139,116.91,57.91,1,0,0,68.4000000000000000000000,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,SKIN,AHC,MASK - B5 RENO,68.4,null,null,null,0,AHC 水光超膜PDRN第四代B5面膜补水修护,SKIN,AHC,BEAUTY&WELLBEING,天猫超市,KA EC,TMALL SUPER
2026-01-25,725677994,955381365598,5879430741936,TMALL SUPER - UL,64950139,523.18,233.63,4,0,0,68.4000000000000000000000,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,SKIN,AHC,MASK - B5 RENO,273.6,null,null,null,0,AHC 水光超膜PDRN第四代B5面膜补水修护,SKIN,AHC,BEAUTY&WELLBEING,天猫超市,KA EC,TMALL SUPER
2026-01-27,725677994,955381365598,5879430741936,TMALL SUPER - UL,64950139,174.82,115.82,2,0,0,68.4000000000000000000000,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,SKIN,AHC,MASK - B5 RENO,136.8,null,null,null,0,AHC 水光超膜PDRN第四代B5面膜补水修护,SKIN,AHC,BEAUTY&WELLBEING,天猫超市,KA EC,TMALL SUPER
2026-01-14,725677994,955864333754,0,TMALL SUPER - UL,62715780,0.0,0.0,-5,0,0,9.8790833333333333333333,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,SKIN,AHC,B5,-49.39541666666667,null,null,null,0,null,null,null,null,天猫超市,KA EC,TMALL SUPER
2026-01-03,725677994,955917896397,5877810592643,TMALL SUPER - UL,69764017,350.96,350.96,4,0,0,50.9734000000000000000000,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,SKIN,AHC,B5,203.8936,180,SKIN AHC B5 180,1,1,AHC舒缓B5洁面,SKIN,AHC,BEAUTY&WELLBEING,天猫超市,KA EC,TMALL SUPER
2026-01-10,725677994,955917896397,5877810592643,TMALL SUPER - UL,69764017,740.07,497.06,7,0,0,50.9734000000000000000000,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,SKIN,AHC,B5,356.8138,180,SKIN AHC B5 180,1,1,AHC舒缓B5洁面,SKIN,AHC,BEAUTY&WELLBEING,天猫超市,KA EC,TMALL SUPER
2026-01-15,725677994,955917896397,5877810592643,TMALL SUPER - UL,69764017,450.05,450.05,7,0,0,50.9734000000000000000000,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,SKIN,AHC,B5,356.8138,180,SKIN AHC B5 180,1,1,AHC舒缓B5洁面,SKIN,AHC,BEAUTY&WELLBEING,天猫超市,KA EC,TMALL SUPER
2026-01-22,725677994,955917896397,5877810592643,TMALL SUPER - UL,69764017,837.1,756.1,10,0,0,50.9734000000000000000000,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,SKIN,AHC,B5,509.734,180,SKIN AHC B5 180,1,1,AHC舒缓B5洁面,SKIN,AHC,BEAUTY&WELLBEING,天猫超市,KA EC,TMALL SUPER


In [0]:
%sql
-- 1.根据店铺名，查询shop_id
-- 2.再用shop_id到tmp表查询明细数据

-- select * from  ${spark.var.catalog.dim}.ddm.dim_store t2
-- where t2.customer like '%TMALL SUPER - UL%'

  select  
    period_key,shop_id,item_id,sku_id,customer,sku_code,gmv_before_refund,gmv,pay_qty,bom_num,is_bundle
    ,listprice,total_listprice,total_bom_num,lp_contri,source_type,platform,bu_name,category_name,brand_name,product_line_group
  from ${spark.var.catalog.tmp}.fin.tmp_trd_total_item_sales_bundle
  where bu_name='BEAUTY&WELLBEING'
  and category_name='HAIR'
  and brand_name='CLEAR'
  and substr(period_key,1,7)='2026-01'
  and shop_id='725677994'

period_key,shop_id,item_id,sku_id,customer,sku_code,gmv_before_refund,gmv,pay_qty,bom_num,is_bundle,listprice,total_listprice,total_bom_num,lp_contri,source_type,platform,bu_name,category_name,brand_name,product_line_group
2026-01-17,725677994,1007788688471,6003122909468,TMALL SUPER - UL,65078739,74.9,74.9,1,0,0,68.0000000000000000000000,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,HAIR,CLEAR,CALIBER
2026-01-24,725677994,1007788688471,6003122909468,TMALL SUPER - UL,65078739,0.0,0.0,-6,0,0,68.0000000000000000000000,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,HAIR,CLEAR,CALIBER
2026-01-29,725677994,1007788688471,6003122909468,TMALL SUPER - UL,65078739,299.6,299.6,4,0,0,68.0000000000000000000000,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,HAIR,CLEAR,CALIBER
2026-01-30,725677994,1007788688471,6003122909468,TMALL SUPER - UL,65078739,224.7,224.7,3,0,0,68.0000000000000000000000,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,HAIR,CLEAR,CALIBER
2026-01-08,725677994,537655659887,3211500721351,TMALL SUPER - UL,69678171,10316.98,9666.9,316,0,0,39.3433333333333333333333,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,HAIR,CLEAR,CORE
2026-01-11,725677994,537692916610,3210604757129,TMALL SUPER - UL,69680177,538.65,505.64,15,0,0,39.3483333333333333333333,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,HAIR,CLEAR,CORE
2026-01-15,725677994,537692916610,3210604757129,TMALL SUPER - UL,69680177,334.0,266.49,8,0,0,39.3483333333333333333333,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,HAIR,CLEAR,CORE
2026-01-20,725677994,537692916610,3210604757129,TMALL SUPER - UL,69680177,327.65,261.63,9,0,0,39.3483333333333333333333,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,HAIR,CLEAR,CORE
2026-01-24,725677994,537692916610,3210604757129,TMALL SUPER - UL,69680177,133.82,133.82,4,0,0,39.3483333333333333333333,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,HAIR,CLEAR,CORE
2026-01-30,725677994,537692916610,3210604757129,TMALL SUPER - UL,69680177,392.2,392.2,12,0,0,39.3483333333333333333333,0.0,0.0,1.0,PLATFORM_B2C,TMALL SUPER,BEAUTY&WELLBEING,HAIR,CLEAR,CORE


In [0]:
%sql

-- select * from ${spark.var.catalog.tmp}.fin.tmp_trd_total_item_sales_all
-- where customer like '%PDD - AHC DTC%'
select * from prod_ads.fin_cpt.ads_second_total_sales_mf
where store_name like '%PDD - AHC DTC%'

--   select
--     period_key,shop_id,item_id,sku_id,customer,sku_code,new_gmv_before_refund as gmv_before_refund,new_gmv as gmv,pay_qty,bom_num,is_bundle
--     ,listprice,total_listprice,total_bom_num,lp_contri,source_type,platform,bu_name,category_name,brand_name,product_line_group
--   from ${spark.var.catalog.tmp}.fin.tmp_trd_total_item_sales_bundle
--   where customer like '%PDD - AHC%'

-- select * from ${spark.var.catalog.ads}.fin_cpt.ads_secondary_gsv_gmv_mf
-- where store_name like '%PDD - AHC%'

    -- select  distinct upper(trim(customer)) as customer,shop_id
    -- from ${spark.var.catalog.dim}.ddm.dim_store
    -- where status = 1  and shop_id <> ''
    -- and customer like '%PDD%'

-- select * from  ${spark.var.catalog.dws}.ddm.dws_trd_total_sku_sales_df a 
-- where shop_id='484559035'

-- select * from ${spark.var.catalog.tmp}.fin.tmp_trd_total_sku_sales_source
-- where customer='PDD - AHC'

-- select * from ${spark.var.catalog.tmp}.fin.tmp_trd_total_item_sales 
-- where customer='PDD - AHC'

-- select * from ${spark.var.catalog.tmp}.fin.tmp_trd_total_item_sales_bundle
-- where customer='PDD - AHC'
-- where customer='PDD - AHC DTC'

-- select * from ${spark.var.catalog.tmp}.fin.tmp_trd_total_item_sales_all
-- where customer='PDD - AHC'

-- select * from ${spark.var.catalog.ads}.fin_cpt.ads_secondary_gsv_gmv_mf
-- where store_name like '%PDD - AHC%'
limit 10
-- PDD - AHC DTC	484559035
-- PDD - AHC	869101267

month,bu_code,bu_name,customer_type_code,customer_type,site_code,site_name,sales_type,store_code,store_name,category_code,category_name,brand_code,brand_name,product_line_group_code,product_line_group,secondary_after_gmv_amount,secondary_gmv_amount,create_by,dw_create_time,dw_last_update_time,dw_batch_num,dt
202603,,BEAUTY&WELLBEING,,EC,,PDD,2C,,PDD - AHC DTC,,SKIN,,AHC,,OTHERS,462554.340000,462554.340000,Automatic,2026-11-09T18:30:58Z,2026-11-09T18:30:58Z,20261109183058,2026-11-09
202606,,BEAUTY&WELLBEING,,EC,,PDD,2C,,PDD - AHC DTC,,SKIN,,AHC,,OTHERS,719671.120000,719671.120000,Automatic,2026-11-09T18:30:58Z,2026-11-09T18:30:58Z,20261109183058,2026-11-09
202609,,BEAUTY&WELLBEING,,EC,,PDD,2C,,PDD - AHC DTC,,SKIN,,AHC,,OTHERS,652803.390000,652803.390000,Automatic,2026-11-09T18:30:58Z,2026-11-09T18:30:58Z,20261109183058,2026-11-09
202602,,BEAUTY&WELLBEING,,EC,,PDD,2C,,PDD - AHC DTC,,SKIN,,AHC,,OTHERS,492911.360000,492911.360000,Automatic,2026-11-09T18:30:58Z,2026-11-09T18:30:58Z,20261109183058,2026-11-09
202610,,BEAUTY&WELLBEING,,EC,,PDD,2C,,PDD - AHC DTC,,SKIN,,AHC,,OTHERS,772252.640000,772252.640000,Automatic,2026-11-09T18:30:58Z,2026-11-09T18:30:58Z,20261109183058,2026-11-09
202605,,BEAUTY&WELLBEING,,EC,,PDD,2C,,PDD - AHC DTC,,SKIN,,AHC,,OTHERS,585353.060000,585353.060000,Automatic,2026-11-09T18:30:58Z,2026-11-09T18:30:58Z,20261109183058,2026-11-09
202608,,BEAUTY&WELLBEING,,EC,,PDD,2C,,PDD - AHC DTC,,SKIN,,AHC,,OTHERS,564402.340000,564402.340000,Automatic,2026-11-09T18:30:58Z,2026-11-09T18:30:58Z,20261109183058,2026-11-09
202604,,BEAUTY&WELLBEING,,EC,,PDD,2C,,PDD - AHC DTC,,SKIN,,AHC,,OTHERS,418242.110000,418242.110000,Automatic,2026-11-09T18:30:58Z,2026-11-09T18:30:58Z,20261109183058,2026-11-09
202601,,BEAUTY&WELLBEING,,EC,,PDD,2C,,PDD - AHC DTC,,SKIN,,AHC,,OTHERS,982772.240000,982772.240000,Automatic,2026-11-09T18:30:58Z,2026-11-09T18:30:58Z,20261109183058,2026-11-09
202607,,BEAUTY&WELLBEING,,EC,,PDD,2C,,PDD - AHC DTC,,SKIN,,AHC,,OTHERS,197870.200000,197870.200000,Automatic,2026-11-09T18:30:58Z,2026-11-09T18:30:58Z,20261109183058,2026-11-09


In [0]:
%sql

    select *,1 as flag
    from ${spark.var.catalog.dim}.ddm.dim_prd_sku   -- 打标表
    where is_current = 1
    -- and item_id like 
    and item_id in ('PDD_AHC_DTC','PDD_AHC_NON_DTC')

platform,bu,bu_short,category,brand,shop_id,shop_name,item_id,item_name,sku_id,sku_name,sku_product,product_line_group,is_sample,pack_size,ttl_volume,ttl_volume_unit,remark,start_time,end_time,is_current,src_create_time,src_update_time,etl_create_time,etl_update_time,dt,flag
PDD,BEAUTY&WELLBEING,B&W,SKIN,AHC,869101267,AHC美妆官方旗舰店,PDD_AHC_NON_DTC,PDD_AHC_NON_DTC,null,null,OTHERS,OTHERS,0,null,null,null,null,0000-01-01,9999-12-31,1,2026-06-03 13:34:14.086+08,2026-06-03 13:34:14.086+08,2026-11-10 15:53:55.564901,2026-11-10 15:53:55.564901,20261109,1
PDD,BEAUTY&WELLBEING,B&W,SKIN,AHC,484559035,AHC爱和纯官方旗舰店,PDD_AHC_DTC,PDD_AHC_DTC,null,null,OTHERS,OTHERS,0,null,null,null,null,0000-01-01,9999-12-31,1,2026-06-03 13:34:14.086+08,2026-06-03 13:34:14.086+08,2026-11-10 15:53:55.564901,2026-11-10 15:53:55.564901,20261109,1


In [0]:
%sql
select 
  item_id,period_key
  ,gmv   as  gmv_before_refund
  ,gmv_after_refund as gmv
  ,case when a.platform = 'PDD' then buyer_cnt else qty end as qty   -- 拼多多 使用 支付买家数
  ,a.shop_id
  ,shop_name,sku_id,sku_name
  ,if(a.period_key > '2025-10-16' and a.shop_id = '621699362', 'PDD - VSL', d.customer) as customer     -- 20251204修改
  ,platform
  ,source_type
from ${spark.var.catalog.dws}.ddm.dws_trd_total_sku_sales_df a 
left join 
(
    select  distinct upper(trim(customer)) as customer,shop_id
    from ${spark.var.catalog.dim}.ddm.dim_store
    where status = 1  and shop_id <> '' and shop_id='484559035'
) d 
on nvl(a.shop_id,'') = nvl(d.shop_id,'')
where a.period_type='日' and left(a.period_key,7) >= '$start_mth' 
      and left(a.period_key,7) <= '$end_mth'
      and a.platform not in ('JD','VIP','TMALL SUPER')
      and d.customer <> 'TMALL FGS - AHC CBE' and d.customer <> 'TMALL FGS - AHC'

item_id,period_key,gmv_before_refund,gmv,qty,shop_id,shop_name,sku_id,sku_name,customer,platform,source_type
PDD_AHC_DTC,2026-02-03,14642.5,14642.5,173,484559035,AHC爱和纯官方旗舰店,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,PDD,null
PDD_AHC_DTC,2026-07-10,2432.0,2432.0,18,484559035,AHC爱和纯官方旗舰店,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,PDD,null
PDD_AHC_DTC,2026-07-23,2588.9,2588.9,25,484559035,AHC爱和纯官方旗舰店,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,PDD,null
PDD_AHC_DTC,2026-09-22,21029.88,21029.88,188,484559035,AHC爱和纯官方旗舰店,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,PDD,null
PDD_AHC_DTC,2026-02-12,16680.0,16680.0,180,484559035,AHC爱和纯官方旗舰店,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,PDD,null
PDD_AHC_DTC,2026-03-23,17445.9,17445.9,222,484559035,AHC爱和纯官方旗舰店,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,PDD,null
PDD_AHC_DTC,2026-07-05,19432.8,19432.8,203,484559035,AHC爱和纯官方旗舰店,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,PDD,null
PDD_AHC_DTC,2026-07-31,2419.4,2419.4,20,484559035,AHC爱和纯官方旗舰店,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,PDD,null
PDD_AHC_DTC,2026-07-25,1043.3,1043.3,14,484559035,AHC爱和纯官方旗舰店,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,PDD,null
PDD_AHC_DTC,2026-03-29,11934.25,11934.25,85,484559035,AHC爱和纯官方旗舰店,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,PDD,null


In [0]:
%sql

    select 
      period_key
      -- ,customer
      ,case when customer='JD - REGENERATE' then 'JD - UL'  else customer  end as customer --20260907
      ,bu
      ,category
      ,brand
      ,product_line_group
      ,case when flag = 1 then 'Y' else 'N' end as jz_flag  --    `是否有机制打标`
      ,case when sku_code = '' or sku_code is null then 'N' else 'Y' end as sku_flag -- `是否有SKU Mapping`
      ,case when flag = 1 and sku_code <> '' and sku_code is not null then 'Y' else 'N' end as both_flag
      ,gmv
    from ${spark.var.catalog.tmp}.fin.tmp_trd_total_item_sales_all
    where abs(gmv) > 0
    and  customer='PDD - AHC'

period_key,customer,bu,category,brand,product_line_group,jz_flag,sku_flag,both_flag,gmv
2026-01-07,PDD - AHC,null,null,null,null,N,N,N,3813.3
2026-01-29,PDD - AHC,null,null,null,null,N,N,N,11753.94
2026-01-31,PDD - AHC,null,null,null,null,N,N,N,7676.92
2026-02-14,PDD - AHC,null,null,null,null,N,N,N,7311.63
2026-02-21,PDD - AHC,null,null,null,null,N,N,N,10212.35
2026-03-01,PDD - AHC,null,null,null,null,N,N,N,6450.5
2026-03-06,PDD - AHC,null,null,null,null,N,N,N,8146.96
2026-03-07,PDD - AHC,null,null,null,null,N,N,N,6856.21
2026-03-15,PDD - AHC,null,null,null,null,N,N,N,13837.95
2026-03-29,PDD - AHC,null,null,null,null,N,N,N,8709.56


In [0]:
%sql

-- select  * from ${spark.var.catalog.ads}.fin_cpt.ads_second_total_sales_mf where store_name='PDD - AHC DTC'  order by dt --and dt='2026-06-06'
select * from ${spark.var.catalog.tmp}.fin.tmp_trd_total_item_sales_all where customer='PDD - AHC DTC'
limit 1000

-- PDD - AHC
-- PDD - AHC DTC


period_key,shop_id,item_id,sku_id,customer,sku_code,gmv_before_refund,gmv,pay_qty,bom_num,is_bundle,listprice,total_listprice,total_bom_num,lp_contri,source_type,platform,bu_name,category_name,brand_name,product_line_group,gsv,pack_size,core_pack_group,is_main_product,flag,item_name,category,brand,bu,shop_name,customer_type,site_name
2026-01-01,484559035,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,null,15160.24,15160.24,149,0,0,0.0001000000000000000000,0.0,0.0,1.0,null,PDD,null,null,null,null,0.0149,null,null,null,0,null,null,null,null,AHC爱和纯官方旗舰店,EC,PDD
2026-01-27,484559035,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,null,31616.29,31616.29,318,0,0,0.0001000000000000000000,0.0,0.0,1.0,null,PDD,null,null,null,null,0.0318,null,null,null,0,null,null,null,null,AHC爱和纯官方旗舰店,EC,PDD
2026-02-04,484559035,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,null,30835.9,30835.9,364,0,0,0.0001000000000000000000,0.0,0.0,1.0,null,PDD,null,null,null,null,0.0364,null,null,null,0,null,null,null,null,AHC爱和纯官方旗舰店,EC,PDD
2026-02-23,484559035,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,null,17730.85,17730.85,216,0,0,0.0001000000000000000000,0.0,0.0,1.0,null,PDD,null,null,null,null,0.0216,null,null,null,0,null,null,null,null,AHC爱和纯官方旗舰店,EC,PDD
2026-05-17,484559035,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,null,14219.48,14219.48,128,0,0,0.0001000000000000000000,0.0,0.0,1.0,null,PDD,null,null,null,null,0.0128,null,null,null,0,null,null,null,null,AHC爱和纯官方旗舰店,EC,PDD
2026-05-18,484559035,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,null,16066.04,16066.04,150,0,0,0.0001000000000000000000,0.0,0.0,1.0,null,PDD,null,null,null,null,0.015000000000000001,null,null,null,0,null,null,null,null,AHC爱和纯官方旗舰店,EC,PDD
2026-06-08,484559035,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,null,24023.1,24023.1,239,0,0,0.0001000000000000000000,0.0,0.0,1.0,null,PDD,null,null,null,null,0.0239,null,null,null,0,null,null,null,null,AHC爱和纯官方旗舰店,EC,PDD
2026-06-29,484559035,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,null,22891.99,22891.99,226,0,0,0.0001000000000000000000,0.0,0.0,1.0,null,PDD,null,null,null,null,0.022600000000000002,null,null,null,0,null,null,null,null,AHC爱和纯官方旗舰店,EC,PDD
2026-07-10,484559035,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,null,2432.0,2432.0,18,0,0,0.0001000000000000000000,0.0,0.0,1.0,null,PDD,null,null,null,null,0.0018000000000000002,null,null,null,0,null,null,null,null,AHC爱和纯官方旗舰店,EC,PDD
2026-07-16,484559035,PDD_AHC_DTC,PDD_AHC_DTC,PDD - AHC DTC,null,1659.93,1659.93,17,0,0,0.0001000000000000000000,0.0,0.0,1.0,null,PDD,null,null,null,null,0.0017000000000000001,null,null,null,0,null,null,null,null,AHC爱和纯官方旗舰店,EC,PDD


In [0]:
%sql

with  ta as (
select 
  item_id,period_key
  ,gmv   as  gmv_before_refund
  ,gmv_after_refund as gmv
  ,case when a.platform = 'PDD' then buyer_cnt else qty end as qty   -- 拼多多 使用 支付买家数
  ,a.shop_id
  ,shop_name,sku_id,sku_name
  ,if(a.period_key > '2025-10-16' and a.shop_id = '621699362', 'PDD - VSL', d.customer) as customer     -- 20251204修改
  ,platform
  ,source_type
from ${spark.var.catalog.dws}.ddm.dws_trd_total_sku_sales_df a 
left join 
(
    select  distinct upper(trim(customer)) as customer,shop_id
    from ${spark.var.catalog.dim}.ddm.dim_store
    where status = 1  and shop_id <> ''
) d 
on nvl(a.shop_id,'') = nvl(d.shop_id,'')
where a.period_type='日' and left(a.period_key,7) >= '$start_mth' 
      and left(a.period_key,7) <= '$end_mth'
      and a.platform not in ('JD','VIP','TMALL SUPER')
      and d.customer <> 'TMALL FGS - AHC CBE' and d.customer <> 'TMALL FGS - AHC'

)

select * from ta 
where customer ='TMALL FGS - BPC'
limit 100

item_id,period_key,gmv_before_refund,gmv,qty,shop_id,shop_name,sku_id,sku_name,customer,platform,source_type
1000553437473,2026-10-09,0.0,0.0,0,1396,联合利华官方旗舰店,6194029770994,:,TMALL FGS - BPC,TMALL FGS,DTC
1000553437473,2026-10-09,0.0,0.0,0,1396,联合利华官方旗舰店,6194004770836,:,TMALL FGS - BPC,TMALL FGS,DTC
1000553437473,2026-10-09,103.23,103.23,1,1396,联合利华官方旗舰店,6021894568196,:,TMALL FGS - BPC,TMALL FGS,DTC
1000553437473,2026-10-09,385.21,385.21,7,1396,联合利华官方旗舰店,6021894568195,:,TMALL FGS - BPC,TMALL FGS,DTC
1001186844997,2026-06-05,97.51,97.51,1,1396,联合利华官方旗舰店,6041044116494,香味:【抗屑控油固发】全擎充能瓶280g*3,TMALL FGS - BPC,TMALL FGS,DTC
1001186844997,2026-06-05,368.12,368.12,3,1396,联合利华官方旗舰店,6170008062415,香味:【抗屑控油固发】全擎充能瓶650g*2,TMALL FGS - BPC,TMALL FGS,DTC
1001186844997,2026-06-05,273.44,273.44,4,1396,联合利华官方旗舰店,6152009227151,香味:【抗屑控油固发】全擎充能瓶650g,TMALL FGS - BPC,TMALL FGS,DTC
1001186844997,2026-06-05,231.55,231.55,6,1396,联合利华官方旗舰店,6041044116493,香味:【抗屑控油固发】全擎充能瓶280g,TMALL FGS - BPC,TMALL FGS,DTC
1001186844997,2026-06-06,270.67,270.67,4,1396,联合利华官方旗舰店,6152009227151,香味:【抗屑控油固发】全擎充能瓶650g,TMALL FGS - BPC,TMALL FGS,DTC
1001186844997,2026-06-06,151.82999999999998,151.82999999999998,4,1396,联合利华官方旗舰店,6041044116493,香味:【抗屑控油固发】全擎充能瓶280g,TMALL FGS - BPC,TMALL FGS,DTC


In [0]:
%sql

select * from ${spark.var.catalog.dim}.cd.dim_item_sku_mapping 
where outer_goods_id like '%PDD_AHC_NON_DTC%'
limit 10

site,store_code,store_name,outer_goods_id,outer_goods_sku_id,goods_sn,tc_goods_sku,is_tc,tc_goods_sl,lastchanged,etl_date,start_date,end_date,source_type
